In [4]:
import os 
import re
import glob
import numpy as np
import pandas as pd

In [5]:
# where the smooth files are located
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"
# Saving CSVs 
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/line_of_sight"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Constants
BEAD_DIAMETER = 0.008  # meters
BEAD_RADIUS = BEAD_DIAMETER / 2

# ==========================================================
# PROCESS
# ==========================================================
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv")))

for path in csv_files:
    fname = os.path.basename(path)
    print(f"Processing LOS: {fname}")
    df = pd.read_csv(path)
    
    # Extract Trial + RPM for filename
    m = re.search(r"(Trial\d+_\d+rpm)", fname)
    trial_name = m.group(1) if m else "Unknown_Trial"

    # Extract points
    head = df[["pt2_X", "pt2_Y", "pt2_Z"]].values
    bead = df[["pt1_X", "pt1_Y", "pt1_Z"]].values
    abdomen_tip = df[["pt7_X", "pt7_Y", "pt7_Z"]].values

    # Vectors
    v = bead - head           # head → bead
    u = head - abdomen_tip    # body axis

    # Distances
    d = np.linalg.norm(v, axis=1)
    p = np.linalg.norm(u, axis=1)

    valid = np.isfinite(d) & np.isfinite(p) & (d > 0) & (p > 0)

    # 1. Angular Error (Theta)
    v_hat = np.full_like(v, np.nan)
    u_hat = np.full_like(u, np.nan)
    v_hat[valid] = v[valid] / d[valid, None]
    u_hat[valid] = u[valid] / p[valid, None]

    cos_theta = np.full_like(d, np.nan)
    cos_theta[valid] = np.sum(u_hat[valid] * v_hat[valid], axis=1)
    perpendicular = np.linalg.norm(np.cross(u_hat, v_hat), axis=1)
    
    theta_deg = np.degrees(np.arctan2(perpendicular, cos_theta))

    # 2. Cone Angle (Alpha)
    alpha = np.full_like(d, np.nan)
    valid_alpha = valid & (d >= BEAD_RADIUS)
    alpha[valid_alpha] = np.arctan(BEAD_RADIUS / d[valid_alpha])
    cone_angle_deg = 2 * np.degrees(alpha)

    # 3. Solid Angle
    solid_angle_sr = np.full_like(alpha, np.nan)
    solid_angle_sr[valid_alpha] = 2 * np.pi * (1 - np.cos(alpha[valid_alpha]))
    solid_angle_deg2 = solid_angle_sr * (180 / np.pi) ** 2

    # Save CSV
    out_df = pd.DataFrame({
        "frame": df["frame"].values if "frame" in df.columns else np.arange(len(df)),
        "distance_to_bead_m": d,
        "angular_error_deg": theta_deg,
        "cone_angle_deg": cone_angle_deg,
        "solid_angle_sr": solid_angle_sr,
        "solid_angle_deg2": solid_angle_deg2
    })

    out_name = f"{trial_name}_line_of_sight.csv"
    out_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)
    print(f"  Saved CSV: {out_name}")

print("\nLine-of-sight CSV processing complete.")

Processing LOS: Trial2_180rpmxyzpts_SMOOTH.csv
  Saved CSV: Trial2_180rpm_line_of_sight.csv
Processing LOS: Trial2_200rpmxyzpts_SMOOTH.csv
  Saved CSV: Trial2_200rpm_line_of_sight.csv
Processing LOS: Trial3_180rpmxyzpts_SMOOTH.csv
  Saved CSV: Trial3_180rpm_line_of_sight.csv
Processing LOS: Trial4_400rpmxyzpts_SMOOTH.csv
  Saved CSV: Trial4_400rpm_line_of_sight.csv
Processing LOS: Trial5_180rpmxyzpts_SMOOTH.csv
  Saved CSV: Trial5_180rpm_line_of_sight.csv
Processing LOS: Trial5_400rpmxyzpts_SMOOTH.csv
  Saved CSV: Trial5_400rpm_line_of_sight.csv
Processing LOS: Trial7_400rpmxyzpts_SMOOTH.csv
  Saved CSV: Trial7_400rpm_line_of_sight.csv

Line-of-sight CSV processing complete.
